# Trabajo Final ICD 2026 - IBM HR Analytics

Objetivo: construir y comparar dos modelos supervisados para predecir si un empleado abandona la empresa (`Attrition`).

Modelos elegidos:

- Regresion Logistica
- Random Forest

El notebook esta organizado por pasos para que el proceso completo se pueda seguir de principio a fin.

In [ ]:
# ============================================
# PASO 0 - Importacion de librerias
# ============================================

from pathlib import Path

# Manipulacion de datos
import numpy as np
import pandas as pd

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Configuracion visual
sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

print("Librerias importadas correctamente")

In [ ]:
# ============================================
# PASO 1 - Carga del dataset
# ============================================

candidate_paths = [
    Path("IBM HR Analytics Employee_TF.csv"),
    Path("../IBM HR Analytics Employee_TF.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
DATA_URL = "https://raw.githubusercontent.com/Camionerou/trabajo_final_icd_2026_project/main/IBM%20HR%20Analytics%20Employee_TF.csv"

if DATA_PATH is not None:
    df = pd.read_csv(DATA_PATH, sep=";")
    FIGURES_DIR = Path("figuras") if Path("IBM HR Analytics Employee_TF.csv").exists() else Path("../figuras")
    print(f"Dataset cargado desde archivo local: {DATA_PATH}")
else:
    df = pd.read_csv(DATA_URL, sep=";")
    FIGURES_DIR = Path("figuras")
    print("Dataset cargado desde GitHub")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_current_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
# Primeras 5 filas del dataset
df.head()

In [ ]:
# Tipos de datos y valores no nulos
df.info()

In [ ]:
# Estadisticas de las columnas numericas
df.describe().T

In [ ]:
# ============================================
# PASO 2 - Exploracion inicial
# ============================================

# Cantidad y porcentaje de valores faltantes
missing_values = (
    df.isna().sum()
    .reset_index(name="Faltantes")
    .rename(columns={"index": "Columna"})
)
missing_values["Porcentaje"] = (missing_values["Faltantes"] / len(df) * 100).round(2)
missing_values = missing_values.query("Faltantes > 0").sort_values("Faltantes", ascending=False)

print("Valores faltantes detectados:")
missing_values

In [ ]:
# Distribucion de la variable objetivo
attrition_distribution = (
    df["Attrition"]
    .value_counts()
    .rename_axis("Attrition")
    .reset_index(name="Cantidad")
)
attrition_distribution["Porcentaje"] = (attrition_distribution["Cantidad"] / len(df) * 100).round(2)

print("Distribucion de Attrition:")
attrition_distribution

In [ ]:
# Columnas con un unico valor
constant_columns = [column for column in df.columns if df[column].nunique(dropna=False) == 1]

print("Columnas constantes:")
print(constant_columns)
print("\nEstas columnas no ayudan a clasificar porque no varian entre empleados.")

In [ ]:
# Variables categoricas disponibles
categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Columnas categoricas:")
print(categorical_columns)

In [ ]:
# ============================================
# PASO 3 - Graficos exploratorios
# ============================================

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Attrition", color="#4C78A8")
plt.title("Distribucion de la variable objetivo")
plt.xlabel("Rotacion laboral")
plt.ylabel("Cantidad de empleados")
save_current_figure("01_distribucion_attrition.png")
plt.show()

In [ ]:
# Grafico obligatorio: caja y bigotes
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome")
plt.title("Boxplot de salario mensual segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Ingreso mensual")
save_current_figure("02_boxplot_monthly_income_attrition.png")
plt.show()

In [ ]:
# Grafico obligatorio: violin plot
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x="Attrition", y="Age")
plt.title("Violin plot de edad segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Edad")
save_current_figure("03_violin_age_attrition.png")
plt.show()

In [ ]:
# Funcion auxiliar para comparar tasas de rotacion por categoria
def plot_attrition_rate(column, filename):
    rate = (
        df.groupby(column)["Attrition"]
        .apply(lambda values: (values == "Yes").mean() * 100)
        .sort_values(ascending=False)
        .reset_index(name="Porcentaje_Attrition_Yes")
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=rate, x=column, y="Porcentaje_Attrition_Yes", color="#4C78A8")
    plt.title(f"Porcentaje de rotacion segun {column}")
    plt.xlabel(column)
    plt.ylabel("Rotacion Yes (%)")
    plt.xticks(rotation=35, ha="right")
    save_current_figure(filename)
    plt.show()

    return rate

plot_attrition_rate("OverTime", "04_rotacion_por_overtime.png")

In [ ]:
# Tasa de rotacion por rol laboral
plot_attrition_rate("JobRole", "05_rotacion_por_jobrole.png")

In [ ]:
# Mapa de calor de correlaciones numericas
numeric_df = df.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Mapa de calor de correlaciones numericas")
save_current_figure("06_heatmap_correlaciones.png")
plt.show()

In [ ]:
# ============================================
# PASO 4 - Limpieza y preparacion de datos
# ============================================

target = "Attrition"
columns_to_drop = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]

# EmployeeNumber es un identificador.
# Las otras tres columnas se eliminan porque tienen un unico valor.
X = df.drop(columns=[target] + columns_to_drop)
y = df[target].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

print(f"Variables numericas: {len(numeric_features)}")
print(numeric_features)
print(f"\nVariables categoricas: {len(categorical_features)}")
print(categorical_features)

In [ ]:
# Division en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train: {len(X_train)} filas")
print(f"Test:  {len(X_test)} filas")
print("\nProporcion de Attrition = Yes:")
print(f"  Train: {y_train.mean():.2%}")
print(f"  Test:  {y_test.mean():.2%}")
print(f"  Total: {y.mean():.2%}")

In [ ]:
# Transformaciones para preparar los datos antes del modelado
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Preprocesamiento definido correctamente")

In [ ]:
# ============================================
# PASO 5 - Entrenamiento de modelos
# ============================================

models = {
    "Regresion Logistica": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ]
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Modelo entrenado: {name}")

In [ ]:
# Comparar algunas predicciones contra los valores reales
comparacion = pd.DataFrame({
    "Real": y_test.map({0: "No", 1: "Yes"}).values[:10],
})

for name, model in models.items():
    comparacion[name] = pd.Series(model.predict(X_test)[:10]).map({0: "No", 1: "Yes"}).values

comparacion

In [ ]:
# ============================================
# PASO 6 - Evaluacion de modelos
# ============================================

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, y_pred)
    precision_yes = precision_score(y_test, y_pred, zero_division=0)
    recall_yes = recall_score(y_test, y_pred, zero_division=0)
    f1_yes = f1_score(y_test, y_pred, zero_division=0)

    results.append(
        {
            "Modelo": name,
            "Train_Accuracy": train_acc,
            "Accuracy": test_acc,
            "Precision_Yes": precision_yes,
            "Recall_Yes": recall_yes,
            "F1_Yes": f1_yes,
        }
    )

    print("=" * 60)
    print(name)
    print(f"Train Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"Test Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
    print("\nReporte de clasificacion:")
    print(classification_report(y_test, y_pred, target_names=["No", "Yes"], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print("Matriz de confusion:")
    print(cm)
    print(f"Verdaderos Negativos (TN): {tn}")
    print(f"Falsos Positivos (FP):     {fp}")
    print(f"Falsos Negativos (FN):     {fn}")
    print(f"Verdaderos Positivos (TP): {tp}")

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(f"Matriz de confusion - {name}")
    plt.xlabel("Prediccion")
    plt.ylabel("Valor real")
    save_current_figure(f"07_matriz_confusion_{name.lower().replace(' ', '_')}.png")
    plt.show()

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df

In [ ]:
# Comparacion grafica de metricas
results_long = results_df.melt(id_vars="Modelo", var_name="Metrica", value_name="Valor")

plt.figure(figsize=(10, 5))
sns.barplot(data=results_long, x="Metrica", y="Valor", hue="Modelo")
plt.title("Comparacion de metricas entre modelos")
plt.xlabel("Metrica")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
save_current_figure("08_comparacion_modelos.png")
plt.show()

In [ ]:
# ============================================
# PASO 7 - Variables mas importantes
# ============================================

random_forest_model = models["Random Forest"]
feature_names = random_forest_model.named_steps["preprocessor"].get_feature_names_out()
importances = random_forest_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({"Variable": feature_names, "Importancia": importances})
    .sort_values("Importancia", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, y="Variable", x="Importancia", color="#4C78A8")
plt.title("Variables mas importantes segun Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
save_current_figure("09_importancia_variables_random_forest.png")
plt.show()

feature_importance

In [ ]:
# ============================================
# RESUMEN FINAL DEL PROYECTO
# ============================================

best_accuracy = results_df.iloc[0]
best_recall = results_df.sort_values("Recall_Yes", ascending=False).iloc[0]

print("=" * 60)
print("RESUMEN - IBM HR ANALYTICS")
print("=" * 60)
print(f"Dataset: {len(df)} filas x {df.shape[1]} columnas")
print(f"Train:   {len(X_train)} filas")
print(f"Test:    {len(X_test)} filas")
print("\nModelos entrenados:")
for model_name in models:
    print(f"- {model_name}")
print("\nMejor accuracy general:")
print(f"{best_accuracy['Modelo']}: {best_accuracy['Accuracy']:.4f} ({best_accuracy['Accuracy']*100:.2f}%)")
print("\nMejor deteccion de Attrition = Yes:")
print(f"{best_recall['Modelo']}: recall Yes = {best_recall['Recall_Yes']:.4f} ({best_recall['Recall_Yes']*100:.2f}%)")
print("\nConclusion:")
print("Random Forest logra mayor accuracy, pero Regresion Logistica detecta mejor los casos de rotacion laboral.")
print("=" * 60)